In [1]:
import random
import json
import math
from faker import Faker

# Install faker if not available
try:
    from faker import Faker
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'faker'], check=True)
    from faker import Faker

fake = Faker()
print('✅ Dependencies ready.')

✅ Dependencies ready.


In [3]:
# ── USER INPUTS ──────────────────────────────────────────────
NUM_COURSES  = 5
NUM_STUDENTS = 20
NUM_FACULTY  = 10

# Guard: need at least 2 faculty per course (1 teacher + 1 scrutinizer min)
if NUM_FACULTY < 2:
    raise ValueError('At least 2 faculty members are required.')

print(f'\n📋 Generating data for {NUM_COURSES} courses, '
      f'{NUM_STUDENTS} students, {NUM_FACULTY} faculties...')


📋 Generating data for 5 courses, 20 students, 10 faculties...


## 🏗️ Data Generation

In [4]:
# ── HELPER: scrutinizer mark logic ───────────────────────────
ERROR_RATE = 0.20   # 20 % of entries will have an erroneous scrutinizer mark

def scrutinizer_mark(teacher_marks: dict) -> tuple[float, bool]:
    """
    Returns (mark, is_error).
    Normal  (80 %): mark == sum of teacher marks, clamped to [0, 100].
    Erroneous (20%): mark deviates noticeably from the correct sum,
                     simulating a teacher entry error.
    """
    correct = round(min(100.0, sum(teacher_marks.values())), 2)
    if random.random() < ERROR_RATE:
        # Shift by a meaningful amount (±10–30 marks) to clearly signal an error
        delta = random.choice([-1, 1]) * round(random.uniform(10, 30), 2)
        erroneous = round(max(0.0, min(100.0, correct + delta)), 2)
        return erroneous, True
    return correct, False


# ── 1. FACULTIES ─────────────────────────────────────────────
faculties = [
    {
        'faculty_id': f'FAC{str(i+1).zfill(3)}',
        'name': fake.name()
    }
    for i in range(NUM_FACULTY)
]


# ── 2. STUDENTS ──────────────────────────────────────────────
students = [
    {
        'student_id': f'STU{str(i+1).zfill(4)}',
        'name': fake.name()
    }
    for i in range(NUM_STUDENTS)
]


# ── 3. COURSES + ENROLLMENTS + MARKS ─────────────────────────
all_faculty_ids = [f['faculty_id'] for f in faculties]
all_student_ids = [s['student_id'] for s in students]

courses = []

for i in range(NUM_COURSES):
    course_id   = f'CSE{str(i+1).zfill(3)}'
    course_name = f'CSE{str(i+1).zfill(3)}'

    # -- Pick 1–3 teachers, then exactly 1 scrutinizer from the rest --
    max_teachers = min(3, NUM_FACULTY - 1)   # leave at least 1 for scrutinizer
    # num_teachers = random.randint(1, max_teachers)
    num_teachers = 2

    shuffled     = random.sample(all_faculty_ids, len(all_faculty_ids))
    teachers     = shuffled[:num_teachers]
    scrutinizer  = shuffled[num_teachers]    # exactly 1, different from teachers

    # -- Random enrollment (10 % – 80 % of total students) ----
    min_enroll   = max(1, math.ceil(NUM_STUDENTS * 0.10))
    max_enroll   = max(min_enroll, math.floor(NUM_STUDENTS * 0.80))
    num_enrolled = random.randint(min_enroll, max_enroll)
    enrolled_ids = random.sample(all_student_ids, num_enrolled)

    # -- Generate marks for each enrolled student --------------
    enrollments = []
    for sid in enrolled_ids:
        # Each teacher gives an independent mark in [0, 100]
        # Individual marks are kept small so their sum stays ≤ 100
        max_per_teacher = round(100 / num_teachers, 2)
        teacher_marks = {
            tid: round(random.uniform(0, max_per_teacher), 2)
            for tid in teachers
        }

        # Scrutinizer mark = sum of teacher marks (with 20 % error chance)
        scr_mark, is_error = scrutinizer_mark(teacher_marks)

        enrollments.append({
            'student_id'      : sid,
            'teacher_marks'   : teacher_marks,   # {faculty_id: mark}
            'scrutinizer_mark': {                 # exactly 1 scrutinizer
                'faculty_id': scrutinizer,
                'mark'      : scr_mark,
                'is_error'  : is_error            # True flags a teacher entry mistake
            }
        })

    courses.append({
        'course_id'  : course_id,
        'course_name': course_name,
        'teachers'   : teachers,
        'scrutinizer': scrutinizer,
        'enrollments': enrollments
    })


# ── 4. ASSEMBLE TOP-LEVEL JSON ────────────────────────────────
mock_data = {
    'metadata': {
        'num_courses' : NUM_COURSES,
        'num_students': NUM_STUDENTS,
        'num_faculty' : NUM_FACULTY,
        'error_rate'  : ERROR_RATE
    },
    'faculties': faculties,
    'students' : students,
    'courses'  : courses
}

print('✅ Data generated successfully!')
print(f'   Courses  : {len(courses)}')
print(f'   Students : {len(students)}')
print(f'   Faculties: {len(faculties)}')

✅ Data generated successfully!
   Courses  : 5
   Students : 20
   Faculties: 10


Preview (first course)

In [5]:
# Pretty-print first course as a sanity check
preview = {
    'metadata' : mock_data['metadata'],
    'faculties' : mock_data['faculties'][:3],   # first 3
    'students'  : mock_data['students'][:3],    # first 3
    'courses'   : [
        {
            **mock_data['courses'][0],
            'enrollments': mock_data['courses'][0]['enrollments'][:2]  # first 2 enrollments
        }
    ]
}
print(json.dumps(preview, indent=2))

{
  "metadata": {
    "num_courses": 5,
    "num_students": 20,
    "num_faculty": 10,
    "error_rate": 0.2
  },
  "faculties": [
    {
      "faculty_id": "FAC001",
      "name": "Robert Hopkins"
    },
    {
      "faculty_id": "FAC002",
      "name": "Andrea Ingram"
    },
    {
      "faculty_id": "FAC003",
      "name": "Scott Rogers"
    }
  ],
  "students": [
    {
      "student_id": "STU0001",
      "name": "Victor Lee"
    },
    {
      "student_id": "STU0002",
      "name": "Matthew Black"
    },
    {
      "student_id": "STU0003",
      "name": "Chad Deleon"
    }
  ],
  "courses": [
    {
      "course_id": "CSE001",
      "course_name": "CSE001",
      "teachers": [
        "FAC005",
        "FAC001"
      ],
      "scrutinizer": "FAC002",
      "enrollments": [
        {
          "student_id": "STU0017",
          "teacher_marks": {
            "FAC005": 5.09,
            "FAC001": 44.15
          },
          "scrutinizer_mark": {
            "faculty_id": "FAC002",

In [6]:
OUTPUT_FILE = 'academic_mock_data.json'

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(mock_data, f, indent=2, ensure_ascii=False)

import os
size_kb = os.path.getsize(OUTPUT_FILE) / 1024
print(f'✅ Saved → {OUTPUT_FILE}  ({size_kb:.1f} KB)')

✅ Saved → academic_mock_data.json  (16.2 KB)


## 📊 Quick Stats

In [7]:
total_enrollments = sum(len(c['enrollments']) for c in courses)
avg_enrolled = total_enrollments / NUM_COURSES if NUM_COURSES else 0
total_errors  = sum(
    1 for c in courses
    for e in c['enrollments']
    if e['scrutinizer_mark']['is_error']
)
error_pct = (total_errors / total_enrollments * 100) if total_enrollments else 0

print('=' * 52)
print('              DATASET SUMMARY')
print('=' * 52)
print(f'  Total courses            : {NUM_COURSES}')
print(f'  Total students           : {NUM_STUDENTS}')
print(f'  Total faculties          : {NUM_FACULTY}')
print(f'  Total enrollments        : {total_enrollments}')
print(f'  Avg students/course      : {avg_enrolled:.1f}')
print(f'  Scrutinizer errors       : {total_errors} ({error_pct:.1f}%)')
print('=' * 52)

print('\nPer-course breakdown:')
print(f'{"Course":<10} {"Teachers":<10} {"Scrutinizer":<14} {"Students":<10} {"Errors"}')
print('-' * 52)
for c in courses:
    errs = sum(1 for e in c['enrollments'] if e['scrutinizer_mark']['is_error'])
    print(f"{c['course_id']:<10} {len(c['teachers']):<10} "
          f"{c['scrutinizer']:<14} {len(c['enrollments']):<10} {errs}")

              DATASET SUMMARY
  Total courses            : 5
  Total students           : 20
  Total faculties          : 10
  Total enrollments        : 43
  Avg students/course      : 8.6
  Scrutinizer errors       : 4 (9.3%)

Per-course breakdown:
Course     Teachers   Scrutinizer    Students   Errors
----------------------------------------------------
CSE001     2          FAC002         5          1
CSE002     2          FAC008         15         1
CSE003     2          FAC006         4          0
CSE004     2          FAC004         13         2
CSE005     2          FAC002         6          0
